# Visual concepts + difflogic + exact explanations
Run from the extracted project root with the requirements installed. This notebook orchestrates the tested CLI commands. CPU reference backend uses the original vendored difflogic operations; upstream CUDA is optional and untested here.

In [ ]:
import sys, subprocess, json
from pathlib import Path
assert Path("cvpr_logic").is_dir(), "Set the notebook working directory to ltn_pyxai_starter"
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run("-m", "unittest", "cvpr_logic.test_core")

In [ ]:
run("-m", "cvpr_logic.train", "--objective", "bce", "--pyxai-check", "--out", "results/notebook_bce")

In [ ]:
run("-m", "cvpr_logic.train", "--objective", "ltn", "--pyxai-check", "--out", "results/notebook_ltn")

In [ ]:
metrics = json.loads(Path("results/notebook_bce/metrics.json").read_text())
metrics

## Cached features
The next cells create synthetic feature data solely to test the real-data adapter. Replace this NPZ with your verified CUB features. The README defines labels, masks, image groups and all four splits. Use `--precomputed-concepts` if x already contains frozen predicate probabilities.

In [ ]:
run("-m", "cvpr_logic.data", "--out", "results/notebook_features.npz")
run("-m", "cvpr_logic.train", "--data", "results/notebook_features.npz", "--epochs", "20", "--out", "results/notebook_features")

In [ ]:
import numpy as np
with np.load("results/notebook_features.npz", allow_pickle=False) as data:
    keep = data["split"] == "test"
    np.savez_compressed("results/notebook_infer.npz", x=data["x"][keep], ids=data["ids"][keep])
run("-m", "cvpr_logic.infer", "--run", "results/notebook_features", "--data", "results/notebook_infer.npz", "--out", "results/notebook_predictions.jsonl")

## Interpretation
A verified reason is sufficient for the hard circuit. Calibrated premise support supplies a marginal false-issued-premise bound under exchangeability, not conditional accuracy or image-level causal grounding. Examine matched repairs/spoils and predictive quality before claiming improvement. Search budget failures abstain.